In [3]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
diagnose_4_2_4_4_empty_bars.py
--------------------------------
4.2.4.4 の棒グラフが「ほとんど空」になる原因を条件ごとに切り分ける診断ツール。

診断すること：
 1) main 側（figs_OH / figs_FP）の ClusterAssign_* を (mode,index) ごとに検出
 2) signless 側（STEP3.2_signlessCorr_MDS_YYYYmmdd/{OH,FP}/cluster_labels_*.csv）の linear_* 有無を確認
 3) (mode,index) ごとの kA, kB, 共通ID数(n)、除外理由タグを付与
 4) 「k≤30 フィルタ」あり/なしの両ケースで、本文棒グラフに残る条件がどれかを一覧化

出力:
  ROOT/paper_4.2.4.4_OHFP/diagnostics/
    - diag_summary_main_text.csv   ← k≤30 条件での採否と原因
    - diag_summary_all_k.csv       ← all-k 条件での採否と原因
    - diag_missing_by_side.csv     ← main / signless どちらが欠けているか
    - diag_counts.txt              ← コンソール要約をテキスト保存

使い方:
  python diagnose_4_2_4_4_empty_bars.py
  python diagnose_4_2_4_4_empty_bars.py --all-k   # kフィルタ無効で再診断

"""

from pathlib import Path
import os, re, glob
import argparse
import pandas as pd
import numpy as np
from sklearn.metrics import adjusted_rand_score

# ========= 設定 =========
ROOT = Path("/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No")
FIGS_OH = ROOT / "figs_OH"
FIGS_FP = ROOT / "figs_FP"

SIGNLESS_BASE = ROOT / "STEP3.2_signlessCorr_MDS_20250904"  # 環境に合わせて調整
SIGNLESS_DIR = {"OH": SIGNLESS_BASE / "OH", "FP": SIGNLESS_BASE / "FP"}

OUTDIR = ROOT / "paper_4.2.4.4_OHFP" / "diagnostics"
OUTDIR.mkdir(parents=True, exist_ok=True)

DIMS    = ["top3","cumeig"]
INDICES = ["silhouette","dunn","gap","ch","db","ptbiserial"]
K_MAX_KEEP = 30

FN_RE = re.compile(
    r"^ClusterAssign_(?P<mode>top3|cumeig)_(?P<index>silhouette|dunn|gap|ch|db|ptbiserial)_(?P<ds>OH|FP)_(?P<date>\d{8})_(?P<hm>\d{4})\.csv$"
)

# ========= ユーティリティ =========
def read_csv_quiet(path: Path) -> pd.DataFrame | None:
    if not path or not path.exists(): return None
    for enc in ("utf-8","utf-8-sig","cp932"):
        try:
            return pd.read_csv(path, encoding=enc)
        except Exception:
            continue
    return None

def latest_by_mode_index(fig_dir: Path, ds: str):
    """(mode,index)->最新ファイルPath"""
    latest = {}
    for p in fig_dir.glob("ClusterAssign_*.csv"):
        m = FN_RE.match(p.name)
        if not m or m["ds"] != ds:
            continue
        key = (m["mode"], m["index"])
        ts  = f"{m['date']}_{m['hm']}"
        if key not in latest or ts > latest[key][0]:
            latest[key] = (ts, p)
    return {k: v[1] for k,v in latest.items()}

def read_var_cluster(path: Path) -> pd.Series | None:
    df = read_csv_quiet(path)
    if df is None or df.empty: return None
    cols = {c.lower(): c for c in df.columns}
    vcol = cols.get("variable", df.columns[0])
    ccol = cols.get("cluster",  df.columns[-1])
    s = pd.Series(df[ccol].values, index=df[vcol].astype(str).values, name="cluster")
    try: s = s.astype(int)
    except: pass
    return s

def load_signless_labels(signless_dir: Path) -> pd.DataFrame | None:
    """cluster_labels_* の linear_* 列だけ保持"""
    cands = sorted(signless_dir.glob("cluster_labels_*.csv"), key=os.path.getmtime, reverse=True)
    if not cands: return None
    df = read_csv_quiet(cands[0])
    if df is None or "id" not in df.columns: return None
    keep = [c for c in df.columns if str(c).startswith("linear_")]
    if not keep: return None
    out = df[["id"]+keep].copy()
    out.columns = ["id"] + [c.replace("linear_","",1) for c in keep]
    return out

def k_from_series(s: pd.Series) -> int:
    if s is None or s.empty: return np.nan
    return int(pd.Series(s).nunique(dropna=True))

def diag_once(k_filter_on: bool = True) -> pd.DataFrame:
    """k≤30フィルタの有無を切り替えて診断"""
    oh_latest = latest_by_mode_index(FIGS_OH, "OH")
    fp_latest = latest_by_mode_index(FIGS_FP, "FP")
    keys = sorted(list(set(oh_latest.keys()).intersection(fp_latest.keys())))
    rows = []

    # signless availability
    S_OH = load_signless_labels(SIGNLESS_DIR["OH"])
    S_FP = load_signless_labels(SIGNLESS_DIR["FP"])
    signless_avail = set()
    if S_OH is not None:
        signless_avail.update([c for c in S_OH.columns if c != "id"])
    if S_FP is not None:
        signless_avail.intersection_update([c for c in S_FP.columns if c != "id"]) if signless_avail else signless_avail.update([c for c in S_FP.columns if c != "id"])

    for mode, index in keys:
        # main: OH/FP それぞれの ClusterAssign を読む
        sOH = read_var_cluster(oh_latest[(mode,index)])
        sFP = read_var_cluster(fp_latest[(mode,index)])

        # k（クラスタ数）
        kOH = k_from_series(sOH)
        kFP = k_from_series(sFP)

        # signless 対応の有無（同じ condition 名が linear_* に存在するか）
        cond = f"{mode}_{index}"
        has_signless = cond in signless_avail

        # 本来の棒グラフ作成では、OH と FP で「new vs signless（線形）」を突き合わせるため、
        # まず “new(=main)” 同士の共通IDを確認（ここでは Variable 名一致を proxy に使う）
        if sOH is not None and sFP is not None:
            ids_common = sorted(set(sOH.index).intersection(set(sFP.index)))
            n_common_vars = len(ids_common)
        else:
            n_common_vars = 0

        # kフィルタに引っかかるか？
        filtered_by_k = False
        if k_filter_on and ( (not np.isnan(kOH) and kOH > K_MAX_KEEP) or (not np.isnan(kFP) and kFP > K_MAX_KEEP) ):
            filtered_by_k = True

        # タグ付け（最も強い原因を上に）
        reasons = []
        if sOH is None or sFP is None:
            # どちらが無いか詳細を記録
            if sOH is None: reasons.append("NO_MAIN_FILE_OH")
            if sFP is None: reasons.append("NO_MAIN_FILE_FP")
        if not has_signless:
            reasons.append("NO_SIGNLESS_FILE")
        if filtered_by_k:
            reasons.append("K_FILTERED_OUT")
        if n_common_vars < 2:
            reasons.append("LOW_MATCH_N")

        # 「本文に残る（OK_INCLUDED）」か？
        ok_included = False
        if (sOH is not None) and (sFP is not None) and has_signless and (not filtered_by_k) and (n_common_vars >= 2):
            ok_included = True
        if ok_included and not reasons:
            reasons = ["OK_INCLUDED"]

        rows.append({
            "mode": mode,
            "index": index,
            "kOH": kOH,
            "kFP": kFP,
            "has_main_OH": sOH is not None,
            "has_main_FP": sFP is not None,
            "has_signless_same_condition": has_signless,
            "n_common_variables(OH∩FP)": n_common_vars,
            "filtered_by_k<=30": filtered_by_k if k_filter_on else False,
            "status": "|".join(reasons) if reasons else "OK_INCLUDED"
        })

    return pd.DataFrame(rows).sort_values(["mode","index"])

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--all-k", action="store_true", help="kフィルタを無効化（= all-k と同様の条件で診断）")
    args = ap.parse_args()

    # k≤30（本文条件）で診断
    df_main = diag_once(k_filter_on= not args.all_k)
    out1 = OUTDIR / ("diag_summary_main_text.csv" if not args.all_k else "diag_summary_all_k.csv")
    df_main.to_csv(out1, index=False, encoding="utf-8-sig")
    print(f"[SAVE] {out1}")

    # どちらが欠けているのか俯瞰
    miss_rows = []
    for ds, figs in [("OH", FIGS_OH), ("FP", FIGS_FP)]:
        latest = latest_by_mode_index(figs, ds)
        have = set(latest.keys())
        miss = [(m, i) for m in DIMS for i in INDICES if (m, i) not in have]
        for m, i in miss:
            miss_rows.append({"dataset": ds, "mode": m, "index": i, "missing_main_file": True})

    if len(miss_rows):
        miss_df = pd.DataFrame(miss_rows).sort_values(["dataset", "mode", "index"])
    else:
        # 空でもCSVを出す（列だけ維持）
        miss_df = pd.DataFrame(columns=["dataset", "mode", "index", "missing_main_file"])

    out2 = OUTDIR / "diag_missing_by_side.csv"
    miss_df.to_csv(out2, index=False, encoding="utf-8-sig")
    print(f"[SAVE] {out2}")

    # 簡易テキストまとめ（df_mainが空でも安全に）
    if len(df_main):
        n_ok = (df_main["status"] == "OK_INCLUDED").sum()
        n_k  = df_main["status"].str.contains("K_FILTERED_OUT").sum()
        n_no_signless = df_main["status"].str.contains("NO_SIGNLESS_FILE").sum()
        n_low = df_main["status"].str.contains("LOW_MATCH_N").sum()
        n_main_oh = df_main["status"].str.contains("NO_MAIN_FILE_OH").sum()
        n_main_fp = df_main["status"].str.contains("NO_MAIN_FILE_FP").sum()
    else:
        n_ok = n_k = n_no_signless = n_low = n_main_oh = n_main_fp = 0


    summary_text = f"""
[診断まとめ] ({'all-k' if args.all_k else 'k<=30'}) 条件
- OK_INCLUDED: {n_ok}
- K_FILTERED_OUT: {n_k}
- NO_SIGNLESS_FILE: {n_no_signless}
- LOW_MATCH_N: {n_low}
- NO_MAIN_FILE_OH: {n_main_oh}
- NO_MAIN_FILE_FP: {n_main_fp}

詳細は {out1.name} を参照。どの (mode,index) がどの理由で棒から落ちたか一目で分かります。
"""
    print(summary_text)
    (OUTDIR / "diag_counts.txt").write_text(summary_text.strip(), encoding="utf-8")

if __name__ == "__main__":
    import sys
    sys.argv = ["diagnose_4_2_4_4_empty_bars.py"]  # Jupyter引数を無効化
    main()


[SAVE] /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/paper_4.2.4.4_OHFP/diagnostics/diag_summary_main_text.csv
[SAVE] /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/paper_4.2.4.4_OHFP/diagnostics/diag_missing_by_side.csv

[診断まとめ] (k<=30) 条件
- OK_INCLUDED: 4
- K_FILTERED_OUT: 8
- NO_SIGNLESS_FILE: 0
- LOW_MATCH_N: 0
- NO_MAIN_FILE_OH: 0
- NO_MAIN_FILE_FP: 0

詳細は diag_summary_main_text.csv を参照。どの (mode,index) がどの理由で棒から落ちたか一目で分かります。

